# Final step on Kaggle: Task 3, all evaluations, ONNX export, app samples
Attach as **inputs** (right panel -> Input -> Add Input -> Your Work / Notebook Output): the finished **Tasks 1-2** notebook version and the finished **Task 4** notebook version.
Settings: GPU, Internet ON, secret `GH_TOKEN` attached. Use **Save Version -> Save & Run All (Commit)**. Download `models/`, `outputs/eval_v2`, `outputs/eval_t4`, `outputs/t3_v2`, `outputs/samples` afterwards.

In [ ]:
import base64, os, subprocess
BRANCH = 'dev'
REPO = 'AfnanRizwan1/GenAI-A1'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('GH_TOKEN')
except Exception as e:
    TOKEN = None
    print(f'GH_TOKEN secret not available ({type(e).__name__}). Private repo: right panel -> Add-ons -> Secrets -> switch GH_TOKEN on for THIS notebook, then restart.')

def git(*args):
    """Run git; the token is sent as a one-off header, never stored in .git/config or printed."""
    cmd, secrets = ['git'], []
    if TOKEN:
        b64 = base64.b64encode(f'x-access-token:{TOKEN}'.encode()).decode()
        cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {b64}']
        secrets = [TOKEN, b64]
    r = subprocess.run(cmd + list(args), capture_output=True, text=True)
    out = r.stdout + r.stderr
    for s in secrets:
        out = out.replace(s, '***')
    if r.returncode:
        raise RuntimeError(f'git {args[0]} failed:\n{out}')
    return out

os.chdir('/kaggle/working')
if not os.path.exists('GenAI-A1'):
    git('clone', '-b', BRANCH, f'https://github.com/{REPO}.git')
os.chdir('GenAI-A1')
git('pull', 'origin', BRANCH)
print(git('log', '--oneline', '-3'))

In [ ]:
!pip install -q pytorch-msssim optuna mlflow onnx onnxruntime opencv-python-headless scikit-learn gdown
!nvidia-smi | head -12

In [ ]:
# copy results of the attached notebooks into ./outputs: all *_v2 folders (Tasks 1-2) and t4 (Task 4)
import glob, os, shutil
os.makedirs('outputs', exist_ok=True)
def pull(marker, pick):
    hits = glob.glob(f'/kaggle/input/**/{marker}', recursive=True)
    assert hits, f'attach the notebook output that contains {marker}'
    src = os.path.dirname(os.path.dirname(hits[0]))
    for d in sorted(os.listdir(src)):
        if pick(d) and not os.path.exists(f'outputs/{d}'):
            shutil.copytree(os.path.join(src, d), f'outputs/{d}')
pull('t2_cls_v2/best.pt', lambda d: d.endswith('_v2'))
pull('t4/best.pt', lambda d: d == 't4')
print(sorted(os.listdir('outputs')))

In [ ]:
# Oxford-IIIT Pet cache and FS2K (for the Task 4 test evaluation)
!python -c "from src.data.pets import load_pets; d = load_pets('data'); print({k: v.shape for k, v in d.items()})"
import glob, os
if not glob.glob('data/fs2k/**/anno_train.json', recursive=True):
    os.makedirs('data/fs2k', exist_ok=True)
    os.system('gdown 1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp -O data/fs2k.zip && unzip -q data/fs2k.zip -d data/fs2k')
print(glob.glob('data/fs2k/**/anno_*.json', recursive=True))

In [ ]:
# Task 3: Optuna search + joint training of the soft mixture-of-experts
!GPU=0 TAG=_v2 TRIALS=20 TRIAL_EPOCHS=8 FINAL_EPOCHS=40 bash scripts/run_t3.sh

In [ ]:
# test-set evaluation: input baseline, Task 1, Task 2 oracle / predicted routing, Task 3 soft MoE (+ routing analysis)
!python -m src.eval.evaluate --t1 outputs/t1_v2/best.pt --cls outputs/t2_cls_v2/best.pt --specs outputs/spec_salt_v2/best.pt outputs/spec_blur_v2/best.pt outputs/spec_occlusion_v2/best.pt --moe outputs/t3_v2/best.pt --out outputs/eval_v2

In [ ]:
# Task 4 test evaluation on the official FS2K test split
!python -m src.eval.evaluate_t4 --gen outputs/t4/best.pt --data-root data/fs2k --out outputs/eval_t4

In [ ]:
# export every inference model to ONNX and verify against PyTorch (fails if any difference exceeds 1e-4)
!python -m src.export.export_onnx --t1 outputs/t1_v2/best.pt --cls outputs/t2_cls_v2/best.pt --specs outputs/spec_salt_v2/best.pt outputs/spec_blur_v2/best.pt outputs/spec_occlusion_v2/best.pt --moe outputs/t3_v2/best.pt --gen outputs/t4/best.pt --out models
!ls -la models

In [ ]:
# sample pictures for the app (copy them into app/backend/samples in the repository)
!python scripts/make_samples.py --data-root data --out outputs/samples --n 8
!ls outputs/samples